# Teste — Classificação Morfológica do Coreano com Kiwi

O Stanza (testado antes) errou sistematicamente ~27% das palavras em coreano
— principalmente substantivo+partícula virando "advérbio", e verbo conjugado
virando "conjunção subordinativa". O Kiwi é uma ferramenta feita
especificamente pra coreano (licença LGPL v3, também gratuita), e — diferente
do Stanza — **roda 100% local, sem precisar baixar nenhum modelo da internet**.

**Diferença importante**: o Kiwi separa a palavra em pedaços (radical +
partícula + terminação), cada um com sua própria classificação — não
classifica a palavra inteira de uma vez como o Stanza fazia. Por isso esse
notebook mostra os resultados **agrupados por palavra original**, com os
pedaços de dentro de cada uma listados por baixo — assim dá pra ver tanto a
palavra inteira quanto o detalhe de cada peça.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q kiwipiepy pandas

import shutil, sys
from pathlib import Path

from kiwipiepy import Kiwi
import pandas as pd
from google.colab import drive

pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 150)

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# O Kiwi já vem com o modelo embutido no pacote — não precisa baixar nada
kiwi = Kiwi()
print("✅ Kiwi pronto (modelo já embutido, sem download externo)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

print(f"Vídeo: {NOME_ORACAO}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. BAIXAR O SRT "edge" CORREANO DO DRIVE                        ║
# ╚══════════════════════════════════════════════════════════════════╝
from config import PipelineConfig
from drive_utils import DriveClient
from srt_utils import ler_srt

config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE="ko")
drive_client = DriveClient.get()

nome_arquivo = config.nome_srt_edge("ko")
destino_local = Path(nome_arquivo)

ok = drive_client.download(config.pasta_oracao, nome_arquivo, destino_local)
if not ok:
    raise FileNotFoundError(f"Não achei '{nome_arquivo}' em {config.pasta_oracao}")

legendas = ler_srt(destino_local)
TEXTO_COREANO = " ".join(leg.texto for leg in legendas)
print(f"✅ {len(legendas)} bloco(s), {len(TEXTO_COREANO)} caracteres")
print(f"\nInício: {TEXTO_COREANO[:100]}...")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. RODAR O KIWI E AGRUPAR POR PALAVRA ORIGINAL                  ║
# ╚══════════════════════════════════════════════════════════════════╝
resultado = kiwi.analyze(TEXTO_COREANO)
melhor_analise = resultado[0][0]  # a análise mais provável (resultado[0] = top-1, [0] = lista de tokens)

linhas = []
for token in melhor_analise:
    linhas.append({
        "posicao_palavra": token.word_position,
        "peca": token.form,
        "classe_kiwi": token.tag,
        "posicao_frase": token.sent_position,
    })

tabela_pecas = pd.DataFrame(linhas)

# reconstrói a palavra original juntando as peças que têm a mesma posição —
# IMPORTANTE: word_position reinicia a cada frase nova, então precisa
# agrupar por (posicao_frase, posicao_palavra) juntos, não só posicao_palavra
# sozinho (senão a palavra 0 da frase 2 se mistura com a palavra 0 da frase 1)
palavras_reconstruidas = (
    tabela_pecas.groupby(["posicao_frase", "posicao_palavra"])["peca"]
    .apply(lambda pecas: "".join(pecas))
    .rename("palavra_completa")
)
tabela_pecas = tabela_pecas.merge(palavras_reconstruidas, on=["posicao_frase", "posicao_palavra"])

print(f"✅ {len(tabela_pecas)} peça(s) morfológica(s), "
      f"{tabela_pecas['posicao_palavra'].nunique()} palavra(s) original(is)")

# reordena as colunas pra ficar mais fácil de ler: palavra inteira primeiro, depois a peça
tabela_pecas = tabela_pecas[["posicao_palavra", "palavra_completa", "peca", "classe_kiwi", "posicao_frase"]]
tabela_pecas


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. O QUE CADA SIGLA DO KIWI SIGNIFICA (referência rápida)       ║
# ║  Kiwi usa o padrão de tags "Sejong" (o mais usado em PLN pra     ║
# ║  coreano) — aqui só as que mais aparecem nesse texto.            ║
# ╚══════════════════════════════════════════════════════════════════╝
GLOSSARIO_TAGS_KIWI = {
    "NNG": "substantivo comum",
    "NNP": "substantivo próprio",
    "NNB": "substantivo dependente (só existe com outra palavra antes)",
    "NP":  "pronome",
    "NR":  "numeral",
    "VV":  "verbo (ação)",
    "VA":  "verbo descritivo (função de adjetivo)",
    "VX":  "verbo auxiliar",
    "VCP": "cópula ('ser/estar')",
    "MM":  "determinante",
    "MAG": "advérbio",
    "IC":  "interjeição",
    "JKS": "partícula de sujeito",
    "JKC": "partícula de complemento",
    "JKG": "partícula possessiva",
    "JKO": "partícula de objeto",
    "JKB": "partícula adverbial (locativo/direcional/etc)",
    "JKV": "partícula vocativa",
    "JKQ": "partícula de citação",
    "JX":  "partícula auxiliar (tópico, ênfase, etc — inclui 은/는)",
    "JC":  "conjunção (partícula)",
    "EP":  "terminação pré-final (tempo, honorífico)",
    "EF":  "terminação final (fecha a frase)",
    "EC":  "terminação conectiva (liga orações)",
    "ETN": "terminação nominalizadora",
    "ETM": "terminação que transforma verbo em modificador",
    "SF":  "pontuação final (. ! ?)",
    "SP":  "vírgula/pontuação de pausa",
    "XSN": "sufixo formador de substantivo",
    "XSV": "sufixo formador de verbo",
    "XSA": "sufixo formador de verbo descritivo",
    "XSM": "sufixo diverso",
    "XPN": "prefixo formador de substantivo",
    "VV-I": "verbo (raiz irregular)",
    "VV-R": "verbo (raiz regular)",
    "VA-I": "verbo descritivo (raiz irregular)",
    "VCN": "cópula negativa (\'não ser/não estar\')",
    "MAJ": "advérbio conjuntivo",
}

tabela_pecas["significado"] = tabela_pecas["classe_kiwi"].map(GLOSSARIO_TAGS_KIWI).fillna("(não catalogada aqui)")
tabela_pecas


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. RESUMO E EXPORTAÇÃO                                          ║
# ╚══════════════════════════════════════════════════════════════════╝
print("Contagem por classe do Kiwi:")
print(tabela_pecas["classe_kiwi"].value_counts())

nome_csv = f"{NOME_ORACAO}_classificacao_kiwi_coreano.csv"
tabela_pecas.to_csv(nome_csv, index=False, encoding="utf-8-sig")
print(f"\n💾 Salvo em {nome_csv}")

from google.colab import files
files.download(nome_csv)
